# Notebook 07 — Classifier Selection

## Purpose
This notebook performs **classifier selection** to identify the most effective  
scikit-learn classification algorithms for distinguishing **AI-generated** and  
**real** images using the previously extracted **25-dimensional Digital Image  
Processing (DIP) feature vectors**.

Multiple classifiers are evaluated using the same normalized training data so  
that performance can be compared fairly and consistently. Rather than selecting  
a single model, the **top two performing classifiers** are identified and  
carried forward for subsequent training and final independent test evaluation.

---

## Inputs
This notebook uses the normalized training dataset generated by:

- `06_Normalize_and_Prepare_Classifier_Input.ipynb`

The input file is defined via **`project_config.py`** and resolves to:

- `metadata/vectors/train_feature_vectors_normalized.csv`

This dataset contains:

- Metadata columns:
  - `filename`
  - `class_label`
  - `source_dataset`
  - `subset`
- **25 normalized DIP feature columns**

---

## Configuration and Execution Model

This notebook follows the project-wide configuration model:

- all file paths are defined in **`src/project_config.py`**
- no hardcoded paths are used
- outputs are saved using config-defined filenames and directories
- display behavior is controlled using a notebook-level **`VERBOSE` flag**

Execution assumes:

- the repository is available locally or cloned at runtime
- prior pipeline notebooks have completed successfully
- required Python packages (NumPy, pandas, scikit-learn) are installed

No external downloads or preprocessing occur in this notebook.

---

## Important Note
All features have already been normalized using a scaler fit only on the  
training dataset in the previous notebook. No additional normalization  
should be applied here.

The independent test dataset exists, but it is **not used in this notebook**  
and remains reserved for final evaluation.

---

## Process Overview
This notebook evaluates multiple classifier types on the same normalized  
DIP feature set using a consistent cross-validation framework. The workflow includes:

1. loading and validating the normalized training dataset  
2. separating metadata from feature columns  
3. defining candidate classifiers  
4. evaluating baseline performance using stratified k-fold cross-validation  
5. ranking classifiers using multiple performance metrics  
6. applying **controlled (small-grid) hyperparameter tuning** to top-performing models  
7. saving comparison results for reporting and reproducibility  
8. selecting the **top two classifiers** for downstream training and evaluation  

---

## Outputs
This notebook produces (paths defined in `project_config.py`):

- a ranked classifier comparison table for baseline models  
- a ranked comparison of tuned classifier results  

Saved CSV files:

- `metadata/results/classifier_comparison_baseline.csv`
- `metadata/results/classifier_comparison_tuned.csv`

Additional outputs:

- JSON file containing the **top two classifier configurations**
- summary identifying recommended classifiers for subsequent notebooks  

---

## Key Design Choice
This notebook performs **multi-model selection**, not single-model selection.

Rather than committing to a single classifier, the **two best-performing models**  
are selected based on cross-validated ROC-AUC after light hyperparameter tuning.

This design:

- reduces sensitivity to small cross-validation differences  
- enables comparison between different model families  
- strengthens experimental validity in the final evaluation stage  

---

## Classifiers Evaluated
The following classifier types are evaluated:

- Logistic Regression  
- Support Vector Machine (Linear and RBF kernels)  
- Random Forest  
- Extra Trees  
- Gradient Boosting  
- k-Nearest Neighbors (KNN)  
- Gaussian Naive Bayes  
- Multi-Layer Perceptron (MLP)  

---

## Scope Limitation
This notebook does **not** perform:

- exhaustive hyperparameter optimization  
- final model training on full datasets  
- evaluation on the independent test set  

These steps are performed in subsequent notebooks.

---

## Cell-by-Cell Structure

### Cell 1 — Startup
Import libraries, load `project_config.py`, and initialize paths and `VERBOSE`.

### Cell 2 — Load Dataset
Load normalized training feature vectors using config-defined paths.

### Cell 3 — Validate Dataset
Perform fail-safe validation (missing values, schema, class balance, feature count).

### Cell 4 — Prepare Inputs
Separate metadata and features; encode target labels.

### Cell 5 — Define Models
Define candidate classifiers and cross-validation strategy.

### Cell 6 — Baseline Evaluation
Run stratified k-fold cross-validation across all candidate classifiers.

### Cell 7 — Baseline Results
Compile, rank, and optionally display baseline results (controlled by `VERBOSE`).

### Cell 8 — Hyperparameter Tuning
Apply controlled tuning to top-performing classifiers.

### Cell 9 — Save Results
Save baseline/tuned results and top classifier configurations using config paths.

### Cell 10 — Summary
Display final recommendations and saved outputs (minimal output unless `VERBOSE=True`).

In [ ]:
# ============================================
# Cell 1: Startup — Imports, Configuration, and Verification
# ============================================

# -------------------------------------------------
# Standard library imports
# -------------------------------------------------
import os
import sys
import json
import warnings
from pathlib import Path

# -------------------------------------------------
# Third-party imports
# -------------------------------------------------
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate,
    GridSearchCV
)

from sklearn.preprocessing import LabelEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

import time
from itertools import product

# -------------------------------------------------
# Notebook display control
# -------------------------------------------------
VERBOSE = True   # Set to False to reduce detailed output

# -------------------------------------------------
# Suppress warnings for cleaner notebook output
# -------------------------------------------------
if not VERBOSE:
    warnings.filterwarnings("ignore")

# -------------------------------------------------
# Clone repo into Colab runtime if needed
# -------------------------------------------------
REPO_URL = "https://github.com/pgailinas/dip-ai-image-detection.git"
REPO_DIR = Path("/content/dip-ai-image-detection")

if not REPO_DIR.exists():
    print("Cloning repository...")
    os.system(f"git clone {REPO_URL} {REPO_DIR}")

# -------------------------------------------------
# Make src/ importable
# -------------------------------------------------
SRC_DIR = REPO_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# -------------------------------------------------
# Import shared project configuration
# -------------------------------------------------
from project_config import (
    VECTORS_METADATA_DIR,
    RESULTS_METADATA_DIR,
    TRAIN_NORMALIZED_FILENAME,
    CLASSIFIER_COMPARISON_BASELINE_FILENAME,
    CLASSIFIER_COMPARISON_TUNED_FILENAME,
    BEST_MODEL_CONFIG_FILENAME,
    NUM_FEATURES,
    RANDOM_SEED,
)

# -------------------------------------------------
# Define required file paths using config constants
# -------------------------------------------------
TRAIN_FEATURE_VECTORS_NORMALIZED_CSV = Path(VECTORS_METADATA_DIR) / TRAIN_NORMALIZED_FILENAME

BASELINE_RESULTS_CSV = Path(RESULTS_METADATA_DIR) / CLASSIFIER_COMPARISON_BASELINE_FILENAME
TUNED_RESULTS_CSV = Path(RESULTS_METADATA_DIR) / CLASSIFIER_COMPARISON_TUNED_FILENAME
BEST_CLASSIFIER_JSON = Path(RESULTS_METADATA_DIR) / BEST_MODEL_CONFIG_FILENAME

Path(RESULTS_METADATA_DIR).mkdir(parents=True, exist_ok=True)

# -------------------------------------------------
# Verify required input file
# -------------------------------------------------
print("Verifying required normalized training CSV file...\n")

if not TRAIN_FEATURE_VECTORS_NORMALIZED_CSV.exists():
    raise FileNotFoundError(
        f"Missing required file: {TRAIN_FEATURE_VECTORS_NORMALIZED_CSV}"
    )

print("Required normalized training CSV file is present.")

# -------------------------------------------------
# Optional verbose output
# -------------------------------------------------
if VERBOSE:
    print(f"Input file:         {TRAIN_FEATURE_VECTORS_NORMALIZED_CSV}")
    print(f"Output directory:   {RESULTS_METADATA_DIR}")
    print(f"Expected features:  {NUM_FEATURES}")
    print(f"Random seed:        {RANDOM_SEED}")
    print("Libraries imported successfully.")



In [ ]:
# ============================================
# Cell 2: Load Normalized Training Data
# ============================================

print(f"Loading training data from: {TRAIN_FEATURE_VECTORS_NORMALIZED_CSV}")

# -------------------------------------------------
# Load dataset
# -------------------------------------------------
df_train = pd.read_csv(TRAIN_FEATURE_VECTORS_NORMALIZED_CSV)

print("\nTraining dataset loaded successfully.")
print(f"Shape: {df_train.shape}")

# -------------------------------------------------
# Preview data (optional)
# -------------------------------------------------
if VERBOSE:
    print("\nFirst 5 rows:")
    try:
        display(df_train.head())  # Jupyter/Colab
    except:
        print(df_train.head())    # Fallback

# -------------------------------------------------
# Show column names (optional)
# -------------------------------------------------
if VERBOSE:
    print("\nColumn names:")
    for col in df_train.columns:
        print(col)



In [ ]:
# ============================================
# Cell 3: Validate Training Data
# ============================================

print("Performing sanity checks...\n")

# --------------------------------------------
# Check for missing values
# --------------------------------------------
missing_values = df_train.isnull().sum().sum()
print(f"Total missing values: {missing_values}")

if missing_values == 0:
    print("No missing values detected.")
else:
    print("WARNING: Missing values detected.")

# --------------------------------------------
# Check class distribution (optional display)
# --------------------------------------------
class_counts = df_train["class_label"].value_counts()

if VERBOSE:
    print("\nClass distribution:")
    print(class_counts)

# --------------------------------------------
# Verify metadata columns
# --------------------------------------------
expected_metadata_cols = [
    "filename",
    "class_label",
    "source_dataset",
    "subset"
]

print("\nChecking metadata columns...")
for col in expected_metadata_cols:
    if col in df_train.columns:
        if VERBOSE:
            print(f"OK: {col}")
    else:
        raise ValueError(f"Missing required column: {col}")

# --------------------------------------------
# Identify feature columns
# --------------------------------------------
feature_cols = [
    col for col in df_train.columns
    if col not in expected_metadata_cols
]

print(f"\nNumber of feature columns: {len(feature_cols)}")

# --------------------------------------------
# Verify feature count
# --------------------------------------------
EXPECTED_FEATURE_COUNT = NUM_FEATURES

if len(feature_cols) == EXPECTED_FEATURE_COUNT:
    print("Feature count is correct.")
else:
    raise ValueError(
        f"Expected {EXPECTED_FEATURE_COUNT} features, "
        f"found {len(feature_cols)}."
    )

print("\nValidation complete.")



In [ ]:
# ============================================
# Cell 4: Prepare X_train and y_train
# ============================================

print("Preparing feature matrix X and target vector y...\n")

# -------------------------------------------------
# Build X and y
# -------------------------------------------------
X_train = df_train[feature_cols].values
y_train = df_train["class_label"].values

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")

# -------------------------------------------------
# Encode labels
# -------------------------------------------------
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)

if VERBOSE:
    print("\nLabel encoding mapping:")
    for i, label in enumerate(label_encoder.classes_):
        print(f"{label} -> {i}")

# -------------------------------------------------
# Final confirmation (optional detail)
# -------------------------------------------------
if VERBOSE:
    print("\nEncoded class distribution:")
    unique, counts = np.unique(y_train_encoded, return_counts=True)
    for u, c in zip(unique, counts):
        print(f"Class {u}: {c}")

print("\nPreparation complete.")



In [ ]:
# ============================================
# Cell 5: Define Candidate Classifiers
# ============================================

print("Defining candidate classifiers and scoring metrics...\n")

# -------------------------------------------------
# Define cross-validation strategy
# -------------------------------------------------
cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_SEED
)

# -------------------------------------------------
# Define candidate classifiers
# -------------------------------------------------
candidate_classifiers = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=RANDOM_SEED
    ),

    "Linear SVM": SVC(
        kernel="linear",
        probability=True,
        random_state=RANDOM_SEED
    ),

    "RBF SVM": SVC(
        kernel="rbf",
        probability=True,
        random_state=RANDOM_SEED
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=RANDOM_SEED,
        n_jobs=-1
    ),

    "Extra Trees": ExtraTreesClassifier(
        n_estimators=100,
        random_state=RANDOM_SEED,
        n_jobs=-1
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        random_state=RANDOM_SEED
    ),

    "KNN": KNeighborsClassifier(
        n_neighbors=5
    ),

    "Gaussian NB": GaussianNB(),

    "MLP": MLPClassifier(
        hidden_layer_sizes=(64, 32),
        max_iter=300,
        random_state=RANDOM_SEED
    )
}

# -------------------------------------------------
# Define scoring metrics
# -------------------------------------------------
scoring_metrics = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

# -------------------------------------------------
# Optional verbose output
# -------------------------------------------------
if VERBOSE:
    print("Candidate classifiers:")
    for name in candidate_classifiers.keys():
        print(f" - {name}")

    print("\nScoring metrics:")
    for metric in scoring_metrics.keys():
        print(f" - {metric}")

    print("\nCross-validation strategy:")
    print(" - StratifiedKFold")
    print(" - n_splits = 5")
    print(" - shuffle = True")
    print(f" - random_state = {RANDOM_SEED}")

print("\nClassifier setup complete.")



In [ ]:
# ============================================
# Cell 6: Run Baseline Classifier Comparison
# ============================================

print("Running baseline classifier comparison...\n")

# NOTE:
# This cell can take several minutes depending on hardware.
# Performance can be improved by:
# - Using a higher-memory environment (e.g., Colab High-RAM)
# - Running on a machine with more CPU cores
#
# This is due to parallel cross-validation (n_jobs=-1)
# and compute-intensive models such as SVM and MLP.

import time

baseline_results = []
total_models = len(candidate_classifiers)

for i, (name, clf) in enumerate(candidate_classifiers.items(), 1):
    print(f"[{i}/{total_models}] Evaluating: {name}")

    start_time = time.time()

    cv_results = cross_validate(
        estimator=clf,
        X=X_train,
        y=y_train_encoded,
        cv=cv_strategy,
        scoring=scoring_metrics,
        return_train_score=False,
        n_jobs=-1
    )

    elapsed = time.time() - start_time

    # -------------------------------------------------
    # Aggregate results
    # -------------------------------------------------
    result = {
        "classifier": name,
        "accuracy_mean": np.mean(cv_results["test_accuracy"]),
        "accuracy_std": np.std(cv_results["test_accuracy"]),
        "precision_mean": np.mean(cv_results["test_precision"]),
        "precision_std": np.std(cv_results["test_precision"]),
        "recall_mean": np.mean(cv_results["test_recall"]),
        "recall_std": np.std(cv_results["test_recall"]),
        "f1_mean": np.mean(cv_results["test_f1"]),
        "f1_std": np.std(cv_results["test_f1"]),
        "roc_auc_mean": np.mean(cv_results["test_roc_auc"]),
        "roc_auc_std": np.std(cv_results["test_roc_auc"]),
        "fit_time_sec": elapsed
    }

    baseline_results.append(result)

    print(f"Completed: {name} (time: {elapsed:.2f} sec)\n")

print("Baseline classifier comparison complete.")



In [ ]:
# ============================================
# Cell 7: Compile and Rank Results
# ============================================

print("Compiling and ranking baseline classifier results...\n")

# -------------------------------------------------
# Convert results to DataFrame
# -------------------------------------------------
df_baseline_results = pd.DataFrame(baseline_results)

# -------------------------------------------------
# Sort by primary metric (ROC-AUC)
# -------------------------------------------------
df_baseline_results = df_baseline_results.sort_values(
    by="roc_auc_mean",
    ascending=False
).reset_index(drop=True)

# -------------------------------------------------
# Display full results (optional)
# -------------------------------------------------
if VERBOSE:
    print("Baseline classifier comparison (sorted by ROC-AUC):\n")

    try:
        display(df_baseline_results)
    except:
        print(df_baseline_results)

# -------------------------------------------------
# Display top classifiers
# -------------------------------------------------
TOP_K = 3

print(f"\nTop {TOP_K} classifiers based on ROC-AUC:\n")

for i in range(min(TOP_K, len(df_baseline_results))):
    row = df_baseline_results.iloc[i]
    print(
        f"{i+1}. {row['classifier']} | "
        f"ROC-AUC: {row['roc_auc_mean']:.4f} | "
        f"F1: {row['f1_mean']:.4f} | "
        f"Accuracy: {row['accuracy_mean']:.4f}"
    )

print("\nRanking complete.")



In [ ]:
# ============================================
# Cell 8: Tune Top Classifiers
# ============================================

print("Tuning top classifiers...\n")

tuned_results = []

# -------------------------------------------------
# Define parameter grids (small, controlled)
# -------------------------------------------------
param_grids = {
    "RBF SVM": {
        "C": [0.1, 1, 10],
        "gamma": ["scale", 0.1, 0.01]
    },

    "MLP": {
        "hidden_layer_sizes": [(64, 32), (128, 64, 32)],
        "alpha": [0.0001, 0.001],
        "learning_rate_init": [0.001, 0.0005]
    },

    "Random Forest": {
        "n_estimators": [100, 200],
        "max_depth": [None, 10, 20]
    }
}

# -------------------------------------------------
# Select top classifiers from baseline ranking
# -------------------------------------------------
TOP_K = 3
top_classifiers = [
    name for name in df_baseline_results["classifier"].head(TOP_K)
    if name in param_grids
]

print("Selected for tuning:")
for name in top_classifiers:
    print(f" - {name}")
print()

total_models = len(top_classifiers)

# -------------------------------------------------
# Tune each top classifier
# -------------------------------------------------
for i, name in enumerate(top_classifiers, 1):
    print(f"[{i}/{total_models}] Tuning: {name}")

    clf = candidate_classifiers[name]
    param_grid = param_grids[name]

    # Count parameter combinations
    n_combinations = len(list(product(*param_grid.values())))

    if VERBOSE:
        print(f"Parameter combinations: {n_combinations}")

    start_time = time.time()

    grid_search = GridSearchCV(
        estimator=clf,
        param_grid=param_grid,
        scoring="roc_auc",
        cv=cv_strategy,
        verbose=1 if VERBOSE else 0,
        n_jobs=-1
    )

    grid_search.fit(X_train, y_train_encoded)

    elapsed = time.time() - start_time

    best_model = grid_search.best_estimator_
    best_score = grid_search.best_score_

    print(f"Best ROC-AUC: {best_score:.4f}")
    print(f"Best params: {grid_search.best_params_}")
    print(f"Time: {elapsed:.2f} sec\n")

    tuned_results.append({
        "classifier": name,
        "best_roc_auc": best_score,
        "best_params": grid_search.best_params_,
        "fit_time_sec": elapsed
    })

print("Tuning complete.")



In [ ]:
# ============================================
# Cell 9: Save Results
# ============================================

print("Saving classifier comparison results...\n")

# -------------------------------------------------
# Convert tuned results to DataFrame
# -------------------------------------------------
df_tuned_results = pd.DataFrame(tuned_results)

if df_tuned_results.empty:
    raise ValueError("No tuned results found. Cell 8 may not have run successfully.")

# -------------------------------------------------
# Sort tuned results by best ROC-AUC
# -------------------------------------------------
df_tuned_results = df_tuned_results.sort_values(
    by="best_roc_auc",
    ascending=False
).reset_index(drop=True)

# -------------------------------------------------
# Save baseline and tuned comparison tables
# -------------------------------------------------
df_baseline_results.to_csv(BASELINE_RESULTS_CSV, index=False)
df_tuned_results.to_csv(TUNED_RESULTS_CSV, index=False)

print(f"Saved baseline results: {BASELINE_RESULTS_CSV}")
print(f"Saved tuned results:    {TUNED_RESULTS_CSV}")

# -------------------------------------------------
# Save top two tuned classifiers to JSON
# -------------------------------------------------
TOP_K_FINAL = 2

top_models = df_tuned_results.head(TOP_K_FINAL)

top_classifiers_summary = []

for _, row in top_models.iterrows():
    top_classifiers_summary.append({
        "classifier": str(row["classifier"]),
        "roc_auc": float(row["best_roc_auc"]),
        "params": dict(row["best_params"])
    })

with open(BEST_CLASSIFIER_JSON, "w") as f:
    json.dump(top_classifiers_summary, f, indent=4)

print(f"Saved top {TOP_K_FINAL} classifier summaries: {BEST_CLASSIFIER_JSON}")

# -------------------------------------------------
# Display saved results
# -------------------------------------------------
print(f"\nTop {TOP_K_FINAL} tuned classifiers:")
for i, model in enumerate(top_classifiers_summary, 1):
    print(f"{i}. {model['classifier']}")
    print(f"   ROC-AUC: {model['roc_auc']:.4f}")

    if VERBOSE:
        print(f"   Params:  {model['params']}")

print("\nSave complete.")



In [ ]:
VERBOSE = True

In [ ]:
# ============================================
# Cell 10: Summary and Recommendations
# ============================================

print("Summarizing classifier selection results...\n")

import textwrap

WRAP_WIDTH = 88

def wrapped_print(text="", indent=""):
    wrapped = textwrap.fill(
        str(text),
        width=WRAP_WIDTH,
        initial_indent=indent,
        subsequent_indent=indent
    )
    print(wrapped)

# -------------------------------------------------
# Safety checks
# -------------------------------------------------
if df_baseline_results.empty or df_tuned_results.empty:
    raise ValueError("Missing results. Ensure Cells 7 and 9 executed successfully.")

# -------------------------------------------------
# Identify best baseline classifier
# -------------------------------------------------
best_baseline_row = df_baseline_results.iloc[0]

print("Best baseline classifier:")
print(f"  Classifier: {best_baseline_row['classifier']}")
print(f"  ROC-AUC:    {best_baseline_row['roc_auc_mean']:.4f}")
print(f"  F1 Score:   {best_baseline_row['f1_mean']:.4f}")
print(f"  Accuracy:   {best_baseline_row['accuracy_mean']:.4f}")

# -------------------------------------------------
# Identify top tuned classifiers
# -------------------------------------------------
TOP_K_FINAL = 2
top_tuned_rows = df_tuned_results.head(TOP_K_FINAL)

print(f"\nTop {TOP_K_FINAL} tuned classifiers:")
for i, (_, row) in enumerate(top_tuned_rows.iterrows(), 1):
    print(f"  {i}. Classifier: {row['classifier']}")
    print(f"     ROC-AUC:    {row['best_roc_auc']:.4f}")

    if VERBOSE:
        wrapped_print(f"Parameters: {row['best_params']}", indent="     ")

# -------------------------------------------------
# Recommendation for subsequent notebook(s)
# -------------------------------------------------
recommended_classifiers = top_tuned_rows["classifier"].tolist()

print("\nRecommendation:")
wrapped_print(
    "The following classifiers are recommended for subsequent final training "
    "and independent test evaluation:",
    indent="  "
)

for clf in recommended_classifiers:
    print(f"   - {clf}")

# -------------------------------------------------
# Interpretation
# -------------------------------------------------
print("\nInterpretation:")
wrapped_print(
    "Classifier selection was performed using the normalized 25-feature DIP "
    "representation and stratified 5-fold cross-validation on the training "
    "set only. The held-out test set was not used in this notebook and "
    "remains reserved for final independent evaluation.",
    indent="  "
)

if VERBOSE:
    wrapped_print(
        "The baseline comparison identified the strongest classifier candidates, "
        "and light hyperparameter tuning was then applied to the top-performing "
        "models. The top two tuned classifiers are carried forward for final "
        "training and independent test evaluation in later notebooks.",
        indent="  "
    )

print("\nClassifier selection notebook complete.")

